In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, Input, MaxPooling2D, UpSampling2D, Concatenate
from tensorflow.keras.models import Model

IMG_SIZE = (128, 128)
SEQ_LENGTH = 2  # The model looks at 2 previous frames to predict the next one

# 1. Custom Dice Metrics (to handle the 95% background imbalance)
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.cast(K.flatten(y_true), 'float32')
    y_pred_f = K.cast(K.flatten(y_pred), 'float32')
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

# 2. Data Loader for Temporal Sequences
def load_and_sequence_data(input_folder, mask_folder, seq_length):
    # Sort files chronologically so the LSTM tracks the flood spread correctly
    valid_files = sorted([f for f in os.listdir(input_folder) if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    images = []
    masks = []
    
    for filename in valid_files:
        img_path = os.path.join(input_folder, filename)
        mask_path = os.path.join(mask_folder, filename)
        
        if not os.path.exists(mask_path):
            continue
            
        # Load Multicolored Input (X)
        img = cv2.imread(img_path)
        img = cv2.resize(img, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        
        # Load B&W Mask (y)
        mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask_img = cv2.resize(mask_img, IMG_SIZE)
        _, binary_mask = cv2.threshold(mask_img, 127, 1, cv2.THRESH_BINARY)
        binary_mask = np.expand_dims(binary_mask, axis=-1)
        
        images.append(img)
        masks.append(binary_mask)
        
    # Create Temporal Sequences (Look at Frame i and i+1 to predict Mask i+2)
    X_seq, y_seq = [], []
    for i in range(len(images) - seq_length):
        X_seq.append(images[i : i + seq_length])
        y_seq.append(masks[i + seq_length]) # Target is the B&W mask of the NEXT frame
        
    return np.array(X_seq), np.array(y_seq)

# Define paths
train_input_dir = os.path.join("flood_dataset", "train")
test_input_dir = os.path.join("flood_dataset", "test")
mask_dir = os.path.join("masks")

print("Loading and sequencing training data...")
X_train, y_train = load_and_sequence_data(train_input_dir, mask_dir, SEQ_LENGTH)

print("Loading and sequencing testing data...")
X_test, y_test = load_and_sequence_data(test_input_dir, mask_dir, SEQ_LENGTH)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")

Loading and sequencing training data...
Loading and sequencing testing data...
X_train shape: (29, 2, 128, 128, 3) | y_train shape: (29, 128, 128, 1)


In [5]:
# Build ConvLSTM Model for Binary Segmentation
inputs = Input(shape=(SEQ_LENGTH, 128, 128, 3))

# Spatial-Temporal feature extraction
x = ConvLSTM2D(16, (3,3), padding='same', return_sequences=False, activation='relu')(inputs)

# Standard spatial CNN encoder-decoder (Preserving the student's layout)
c1 = Conv2D(32, (3,3), activation='relu', padding='same')(x)
p1 = MaxPooling2D((2,2))(c1)

c2 = Conv2D(64, (3,3), activation='relu', padding='same')(p1)

u1 = UpSampling2D((2,2))(c2)
u1 = Concatenate()([u1, c1])

c3 = Conv2D(32, (3,3), activation='relu', padding='same')(u1)

# CRITICAL CHANGE: Output layer -> 1 channel, sigmoid activation for Binary Mask
outputs = Conv2D(1, (1,1), activation='sigmoid')(c3)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss=dice_loss, metrics=['accuracy', dice_coef])

print("Starting Training...")
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=4, # Smaller batch size because LSTMs consume a lot of RAM
    epochs=20
)

Starting Training...
Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step - accuracy: 0.9595 - dice_coef: 0.0839 - loss: 0.9151 - val_accuracy: 0.9590 - val_dice_coef: 0.0958 - val_loss: 0.9042
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 157ms/step - accuracy: 0.9591 - dice_coef: 0.1830 - loss: 0.8353 - val_accuracy: 0.9647 - val_dice_coef: 0.3114 - val_loss: 0.6886
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - accuracy: 0.9661 - dice_coef: 0.5059 - loss: 0.5054 - val_accuracy: 0.9646 - val_dice_coef: 0.5775 - val_loss: 0.4225
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step - accuracy: 0.9616 - dice_coef: 0.6056 - loss: 0.3804 - val_accuracy: 0.9672 - val_dice_coef: 0.5917 - val_loss: 0.4083
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - accuracy: 0.9650 - dice_coef: 0.6331 - loss: 0.3741 - val_accuracy: 0.9649 - val_dice_coef: 0.6292 - val_loss: 0.3708
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 174ms/step - accuracy: 0.9664 - dice_coef: 0.6481 - loss: 0.3483 - val_accuracy: 0.9626 - 

In [7]:
import tf2onnx

# Input signature: (Batch, Sequence Length, Height, Width, Channels)
input_signature = [tf.TensorSpec([None, SEQ_LENGTH, 128, 128, 3], tf.float32, name='sequence_input')]
onnx_model_path = "CNN_LSTM.onnx"

print("Converting trained ConvLSTM model to ONNX...")
model_proto, _ = tf2onnx.convert.from_keras(
    model, 
    input_signature=input_signature, 
    output_path=onnx_model_path
)

print(f"Success! Model exported to {onnx_model_path}")

Converting trained ConvLSTM model to ONNX...
Success! Model exported to CNN_LSTM.onnx
